## Setup
Reads now come from workspace.bronze tables instead of Parquet paths — this is also what makes the Bronze→Silver connection show up as a real edge in Unity Catalog's lineage graph, rather than two disconnected sets of files.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

def bronze(t):
    return spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{t}")

def to_silver(df, name, comment):
    df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.{name}")
    spark.sql(f"COMMENT ON TABLE {CATALOG}.{SILVER_SCHEMA}.{name} IS '{comment}'")

## Tables that only need deduplication
`competitions`, `clubs`, `countries`, `national_teams`, `club_games`, `game_events`, `game_lineups`, and `transfers` have no known type or date issue at this stage — the only cleaning they need is dropping exact duplicate rows.

In [0]:
passthrough = ["competitions", "clubs", "countries", "national_teams",
               "club_games", "game_events", "game_lineups", "transfers"]

for t in passthrough:
    to_silver(bronze(t).dropDuplicates(), t, f"Deduplicated (full-row) copy of bronze.{t}")

## `players`
Deduped on the real natural key, `player_id` — not a full-row match, since two rows could differ in a free-text field like `agent_name` and still be the same player. `date_of_birth` is cast from `timestamp` down to `date`; this can't introduce new nulls, since it was already a proper temporal type in Bronze, not a string being parsed for the first time.

In [0]:
players = (bronze("players")
           .dropDuplicates(["player_id"])
           .withColumn("date_of_birth", F.to_date("date_of_birth")))
to_silver(players, "players", "Deduplicated on player_id; date_of_birth cast from timestamp to date")
print("players: date_of_birth nulls after cast:",
      players.filter(F.col("date_of_birth").isNull()).count())

## `games`
Deduped on `game_id`. `date` was already typed correctly in Bronze, so this cast is a formality — the null count confirms that.

In [0]:
games = (bronze("games")
         .dropDuplicates(["game_id"])
         .withColumn("date", F.to_date("date")))
to_silver(games, "games", "Deduplicated on game_id; date cast from already-correct date type (formality)")
print("games: date nulls after cast:", games.filter(F.col("date").isNull()).count())

## `player_valuations` and `appearances`
Both already have a correctly-typed `date` in Bronze, same reasoning as `games`. No deduplication here — unlike `players` or `games`, we haven't yet confirmed `appearance_id` is actually a unique key rather than just assumed it; that check happens in the EDA notebook, not here.

In [0]:
player_valuations = bronze("player_valuations").withColumn("date", F.to_date("date"))
to_silver(player_valuations, "player_valuations", "date cast from already-correct date type")

appearances = bronze("appearances").withColumn("date", F.to_date("date"))
to_silver(appearances, "appearances", "date cast from already-correct date type")